# 03 Tune cVAE - fold 1

Input: hcsinhgoethe/ppg-cvae-prepared-fold01 and local inputs/proxy/fold_01 supplied by --input.
Five original candidates are selected by validation only. An additional beta=.01, lambda_hr=0 diagnostic uses the same budget and is excluded from candidate selection.
Proxy real validation MAE is 11.89 bpm; detector MAE is 9.10 bpm. Treat HR scores as exploratory and inspect morphology plus the no-HR control.


In [1]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items


{'DATA_ROOT': '/kaggle/input/ppg-dalia-dataset', 'OUTPUT_ROOT': 'artifacts', 'device': 'cuda', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cu128'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Prepared dataset: https://www.kaggle.com/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01
import shutil
from zipfile import ZipFile
DATASET_SLUG = 'ppg-cvae-prepared-fold01'
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
fold_name = f'fold_{FOLD:02d}'
required_files = ('train.npz', 'val.npz', 'manifest.csv', 'normalization_stats.json', 'preprocessing_config.json')
destination = OUTPUT_ROOT/'prepared'/fold_name

# Discover the mounted dataset rather than assuming Kaggle's directory layout.
if ARTIFACT_INPUT_ROOT:
    dataset_roots = [Path(ARTIFACT_INPUT_ROOT)]
else:
    dataset_roots = sorted(p for p in input_dir.rglob(DATASET_SLUG) if p.is_dir()) if input_dir.exists() else []
if not dataset_roots:
    raise FileNotFoundError(f'Attach hcsinhgoethe/{DATASET_SLUG} with krun --dataset, or set PPGCVAE_INPUT_ROOT.')

prepared_sources = []
for root in dataset_roots:
    if not root.is_dir(): raise FileNotFoundError(root)
    for train_file in root.rglob('train.npz'):
        candidate = train_file.parent
        if candidate.name.startswith('fold_') and candidate.name != fold_name: continue
        if all((candidate/name).is_file() for name in required_files):
            prepared_sources.append(candidate)
if len(prepared_sources) > 1:
    raise ValueError(f'Ambiguous prepared artifact directories: {prepared_sources}; set PPGCVAE_INPUT_ROOT explicitly.')
if prepared_sources:
    prepared_source = prepared_sources[0]
    if prepared_source.resolve() != destination.resolve():
        shutil.copytree(prepared_source, destination, dirs_exist_ok=True)
    print('Imported prepared arrays from:', prepared_source)
else:
    # Also support a dataset whose uploaded ZIP has not been unpacked by Kaggle.
    archives = []
    for root in dataset_roots:
        for archive in root.rglob('*.zip'):
            with ZipFile(archive) as z:
                prefix = f'prepared/{fold_name}/'
                if all(prefix+name in z.namelist() for name in required_files):
                    archives.append(archive)
    if len(archives) != 1:
        raise FileNotFoundError(f'Expected one prepared {fold_name} directory or ZIP in {dataset_roots}; found archives: {archives}')
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archives[0]) as z:
        prefix = f'prepared/{fold_name}/'
        for name in (*required_files, 'test.npz', 'subject_summary.csv', 'hr_coverage.csv'):
            member = prefix+name
            if member in z.namelist():
                with z.open(member) as src, (destination/name).open('wb') as dst:
                    shutil.copyfileobj(src, dst)
    print('Imported prepared arrays from ZIP:', archives[0])
assert all((destination/name).is_file() for name in required_files)
print('Prepared fold directory:', destination)

# Proxy is a differentiable estimate, not a ground-truth HR label.
# Preserve validation quality metadata in every tuning run for interpretation.
proxy_source = Path('inputs/proxy')/fold_name
for name in ('hr_proxy.pt','detector_config.json','hr_proxy_validation_metrics.json','detector_validation_metrics.json'):
    if not (proxy_source/name).is_file():
        raise FileNotFoundError(f'Missing {proxy_source/name}; pass --input inputs/proxy/fold_01 to KRun.')
proxy_destination = OUTPUT_ROOT/'proxy'/fold_name
shutil.copytree(proxy_source,proxy_destination,dirs_exist_ok=True)
import json
proxy_quality = json.loads((proxy_destination/'hr_proxy_validation_metrics.json').read_text())
detector_quality = json.loads((proxy_destination/'detector_validation_metrics.json').read_text())
print('Proxy validation MAE bpm:',proxy_quality['best_val_mae_bpm'])
print('Detector real validation metrics:',detector_quality['selected'])


FoldSubjects(fold_id=1, train_subjects=frozenset({'S11', 'S12', 'S15', 'S9', 'S13', 'S8', 'S10', 'S14', 'S7'}), val_subjects=frozenset({'S6', 'S4', 'S5'}), test_subjects=frozenset({'S1', 'S3', 'S2'}))


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-fold01/prepared/fold_01
Prepared fold directory: artifacts/prepared/fold_01
Proxy validation MAE bpm: 11.888657971351767
Detector real validation metrics: {'prominence': 0.1, 'V': 0.9999155191349159, 'mae_hr': 9.10020403735541, 'A3': 0.4336402804764721}


In [3]:
from ppg_cvae.workflow import tune_fold
from ppg_cvae.utils.io import save_json
# Keep the original five-candidate grid; add one separate no-HR-loss diagnostic.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('This full tuning job expects a GPU; run KRun with --gpu T4.')
quality_dir = OUTPUT_ROOT/'tuning'/f'fold_{FOLD:02d}'
save_json({'proxy':proxy_quality,'detector':detector_quality,
           'interpretation':'exploratory; proxy and detector have substantial error on real validation',
           'source_job':'20261004-050604-c55386',
           'additional_control':{'beta':0.01,'lambda_hr':0.0,'role':'diagnostic_only_excluded_from_selection'}},
          quality_dir/'measurement_quality.json')
best = tune_fold(OUTPUT_ROOT,FOLD,device=device,include_no_hr_control=True)
print('Selected among original five candidates using validation only:',best)
print('Also inspect comparison_with_no_hr_control.csv before final training.')


Starting fold 1, beta=0.01, lambda_hr=0.0, seed=42


Completed: {'beta': 0.01, 'lambda_hr': 0.0, 'all_V_ge_95': True, 'mean_mae_hr': 21.737127693257715, 'mean_A3': 0.12166666666666666, 'val_rmse': 0.5448082089424133}


Starting fold 1, beta=0.001, lambda_hr=1.0, seed=42


Completed: {'beta': 0.001, 'lambda_hr': 1.0, 'all_V_ge_95': True, 'mean_mae_hr': 21.835245727210573, 'mean_A3': 0.08166666666666667, 'val_rmse': 0.5827988982200623}


Starting fold 1, beta=0.01, lambda_hr=0.1, seed=42


...

Starting fold 1, beta=0.01, lambda_hr=5.0, seed=42


Completed: {'beta': 0.01, 'lambda_hr': 5.0, 'all_V_ge_95': True, 'mean_mae_hr': 39.625411819875495, 'mean_A3': 0.07666666666666667, 'val_rmse': 0.833853542804718}


Starting fold 1, beta=0.1, lambda_hr=1.0, seed=42


Completed: {'beta': 0.1, 'lambda_hr': 1.0, 'all_V_ge_95': True, 'mean_mae_hr': 26.830944054810843, 'mean_A3': 0.12333333333333334, 'val_rmse': 0.5870015025138855}


Selected among original five candidates using validation only: {'beta': 0.01, 'lambda_hr': 0.1, 'all_V_ge_95': True, 'mean_mae_hr': 19.693561122192943, 'mean_A3': 0.07833333333333332, 'val_rmse': 0.5547592043876648}
Also inspect comparison_with_no_hr_control.csv before final training.
